# Data Check

## Imports and paths

In [8]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Find the repo root (the folder that contains ReadMe.md)
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "ReadMe.md").exists())
RAW_PATH   = ROOT / "Data/DataMainTable/diabetic_data.csv"
CLEAN_PATH = ROOT / "Data/cleaned_diabetic_data.csv"
FIG = ROOT / "Paper/figures"; FIG.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 60)
sns.set_theme(style="whitegrid", context="notebook")
print("Repo root:", ROOT) # use the training split

Repo root: C:\Users\JanAr\IdeaProjects\Data-Analytics-diabetes-readmission\Data-Analytics-diabetes-readmission


## Load the data

In [9]:
# Only "?" is missing. keep_default_na=False stops "None" (= not tested) becoming NaN.
raw = pd.read_csv(RAW_PATH, na_values="?", keep_default_na=False,
                  dtype={"payer_code": str})
df = pd.read_csv(CLEAN_PATH)
df["readmitted_30d"] = (df["readmitted"] == "<30").astype(int)

print(f"Raw:     {raw.shape[0]:,} rows × {raw.shape[1]} columns")
print(f"Cleaned: {df.shape[0]:,} rows × {df.shape[1]} columns (34 + readmitted_30d)")

Raw:     101,766 rows × 50 columns
Cleaned: 69,987 rows × 35 columns (34 + readmitted_30d)


## Unit of analysis
This cell counts the patients and their encounters. Many patients have more than one encounter, so rows in the raw data are not independent. The cleaned data must have one row for each patient.

In [10]:
visits = raw.groupby("patient_nbr").size()
print(f"Unique patients:                 {visits.size:,}")
print(f"Patients with >1 encounter:      {(visits > 1).sum():,}")
print(f"Maximum encounters for 1 patient: {visits.max()}")
print(f"Cleaned rows per patient (max):  {df.groupby('patient_nbr').size().max()}")

Unique patients:                 71,518
Patients with >1 encounter:      16,773
Maximum encounters for 1 patient: 40
Cleaned rows per patient (max):  1


## Missing values and quality problems
This cell shows the share and number of missing values in each column. It also counts unknown gender values and the share of patients with lab tests. It finds the columns that have one value in all rows, because these columns give no information.

In [12]:
miss = (raw.isna().mean()
        .loc[lambda s: s > 0]
        .sort_values(ascending=False)
        .to_frame("missing_share"))
miss["missing_count"] = raw[miss.index].isna().sum()
display(miss.style.format({"missing_share": "{:.1%}", "missing_count": "{:,}"}))

print("Unknown gender:", (raw["gender"] == "Unknown/Invalid").sum())
print("HbA1c tested:        ", f"{(raw['A1Cresult'] != 'None').mean():.1%}")
print("Glucose serum tested:", f"{(raw['max_glu_serum'] != 'None').mean():.1%}")
single = [c for c in raw.columns if raw[c].nunique(dropna=False) == 1]
print("Columns with one value:", single)

,missing_share,missing_count
weight,96.9%,"98,569"
medical_specialty,49.1%,"49,949"
payer_code,39.6%,"40,256"
race,2.2%,"2,273"
diag_3,1.4%,"1,423"
diag_2,0.4%,358
diag_1,0.0%,21


Unknown gender: 3
HbA1c tested:         16.7%
Glucose serum tested: 5.3%
Columns with one value: ['examide', 'citoglipton']


## Target
This cell compares the readmission classes in the raw data and in the cleaned data. Readmission in less than 30 days is the rare class. Thus, accuracy is not a good measure for the model.

In [13]:
target = pd.DataFrame({
    "raw":     raw["readmitted"].value_counts(normalize=True),
    "cleaned": df["readmitted"].value_counts(normalize=True),
}).loc[["NO", ">30", "<30"]]
display(target.style.format("{:.1%}"))
print(f"Cleaned 30-day readmissions: {df['readmitted_30d'].sum():,} "
      f"({df['readmitted_30d'].mean():.1%})")

,raw,cleaned
readmitted,,
NO,53.9%,59.3%
>30,34.9%,31.8%
<30,11.2%,9.0%


Cleaned 30-day readmissions: 6,285 (9.0%)


## Demographics
This cell shows the age and race of the patients. Most patients are 60 years or older, and most are Caucasian. These facts limit how well the results apply to other groups.

In [14]:
age60 = raw["age"].isin(["[60-70)", "[70-80)", "[80-90)", "[90-100)"]).mean()
print(f"Encounters for patients aged 60+: {age60:.1%}")
display(raw["race"].value_counts(normalize=True).to_frame("share").style.format("{:.1%}"))

Encounters for patients aged 60+: 67.4%


,share
race,
Caucasian,76.5%
AfricanAmerican,19.3%
Hispanic,2.0%
Other,1.5%
Asian,0.6%


## Check against published sources
This cell compares the notebook with values from the UCI dataset page [3] and from Strack et al. [5]. These sources are independent of this project. Small differences are possible, because Strack et al. used a different filter order and rounded their percentages.

In [18]:
final_a1c_tested = (df["A1Cresult"] != "Not_Tested").mean()

pub = pd.DataFrame([
    # check,                          notebook value,                   published, source
    ("Raw encounters",                len(raw),                         101_766, "UCI [3]; Strack [5] Methods"),
    ("Final encounters",              len(df),                          69_984,  "Strack [5] Methods"),
    ("30-day readmissions (final)",   df["readmitted_30d"].sum(),       6_459,   "Strack [5] Table 3"),
    ("HbA1c tested, final (%)",       round(final_a1c_tested*100, 1),   18.4,    "Strack [5] Results"),
    ("Weight missing (%)",            round(raw["weight"].isna().mean()*100, 1), 97,  "Strack [5] Table 1"),
    ("Payer code missing (%)",        round(raw["payer_code"].isna().mean()*100, 1), 40, "Strack [5] Methods"),
], columns=["check", "notebook", "published", "source"]).set_index("check")
pub["difference"] = pub["notebook"] - pub["published"]
display(pub)

,notebook,published,source,difference
check,,,,
Raw encounters,101766.0,101766.0,UCI [3]; Strack [5] Methods,0.0
Final encounters,69987.0,69984.0,Strack [5] Methods,3.0
30-day readmissions (final),6285.0,6459.0,Strack [5] Table 3,-174.0
"HbA1c tested, final (%)",18.4,18.4,Strack [5] Results,0.0
Weight missing (%),96.9,97.0,Strack [5] Table 1,-0.1
Payer code missing (%),39.6,40.0,Strack [5] Methods,-0.4


## Internal checks
This cell checks that the numbers agree with each other. It does not use the report or the published values. The removed rows must add up to the difference between the raw data and the cleaned data, each patient must have one row, and the classes must add up to the total.

A. Read the ID mappings

In [ ]:
def read_ids_mapping(path):
    tables, current, name = {}, [], None
    for line in Path(path).read_text().splitlines():
        if not line.strip(",").strip():          # empty separator row
            if name: tables[name] = pd.DataFrame(current, columns=[name, "description"])
            current, name = [], None
        elif name is None:                       # header row
            name = line.split(",")[0]
        else:
            code, desc = line.split(",", 1)
            current.append((int(code), desc.strip().strip('"')))
    if name: tables[name] = pd.DataFrame(current, columns=[name, "description"])
    return tables

ids = read_ids_mapping(ROOT / "Data/DataMainTable/IDS_mapping.csv")
dd = ids["discharge_disposition_id"]
EXPIRED = dd.loc[dd["description"].str.startswith("Expired"), "discharge_disposition_id"].tolist()
HOSPICE = dd.loc[dd["description"].str.startswith("Hospice"), "discharge_disposition_id"].tolist()
print("Expired codes:", EXPIRED)   # expect [11, 19, 20, 21]
print("Hospice codes:", HOSPICE)   # expect [13, 14]

B. Check

In [22]:
dead_hospice = raw["discharge_disposition_id"].isin(EXPIRED + HOSPICE)
after_dh   = raw[~dead_hospice]
after_first = after_dh.sort_values("encounter_id").drop_duplicates("patient_nbr")
after_gender = after_first[after_first["gender"] != "Unknown/Invalid"]

steps = pd.Series({
    "raw":                 len(raw),
    "after death/hospice": len(after_dh),
    "after first stay":    len(after_first),
    "after gender":        len(after_gender),
    "cleaned file":        len(df),
})
display(steps.to_frame("rows"))

assert len(after_gender) == len(df),              "Recalculated cohort differs from cleaned file"
assert df["patient_nbr"].is_unique,               "A patient has more than one row"
assert df["encounter_id"].is_unique,              "Duplicate encounter"
assert df["readmitted"].value_counts().sum() == len(df), "Target has missing values"
assert set(train.patient_nbr).isdisjoint(test.patient_nbr) if "train" in globals() else True
print("All internal checks passed")

,rows
raw,101766
after death/hospice,99343
after first stay,69990
after gender,69987
cleaned file,69987


All internal checks passed
